# Rehearsal fine-tuning: NLLB (lid218e), GlotLID v3, ConLID

Runs the `new_method` continued-training experiment on the data built by stages 02 and 05:

| Phase | Weights | Training data |
|---|---|---|
| `zero_shot` | Released checkpoint | None |
| `target_only` | Checkpoint + appended `pli_Sinh`/`san_Sinh` rows | `train.csv` (3 target groups) |
| `replay` | Same expanded initialization | `train_mixed_11groups.jsonl` (targets + 8 rehearsal groups) |

The pipeline outputs are copied into `new_method/data/` (the bundle's input folder), so the
protocol signature stays tied to file contents. Completed phases with an unchanged
signature are skipped, so re-running is cheap. If any input file changes, `run_experiment`
refuses to mix results: pass a new `output_dir`.

Requires a C++ compiler (`g++`) only when `new_method/bin/lid_native(.exe)` is missing or stale.
See `new_method/README.md` and `new_method/FINETUNING_METHOD.md` for the method.

In [ ]:
models = ["nllb", "glotlid", "conlid"]
output_dir = None  # None keeps new_method/config.json's output_dir
device = None  # None keeps the config value; "cpu" or "cuda" to override

In [ ]:
import os
import sys
from pathlib import Path

# Auto-resolve the project root if running manually
if not os.path.exists("Makefile") and os.path.exists("../../Makefile"):
    os.chdir("../../")

PIPELINE = Path.cwd()
ROOT = PIPELINE / "new_method"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from lidlab.data import load_config

c = load_config(ROOT / "config.json")
if output_dir:
    c["output_dir"] = output_dir
if device:
    c["device"] = device

from lidlab.data import digest
from lidlab.experiment import run_experiment
from scripts.build_native import build

print("new_method root:", ROOT)
print("Results directory:", ROOT / c["output_dir"])

## 1. Sync pipeline datasets into `new_method/data/`

In [ ]:
import shutil


def first_existing(*candidates):
    for candidate in candidates:
        if (PIPELINE / candidate).exists():
            return candidate
    raise FileNotFoundError(f"None of {candidates} exist under {PIPELINE}")


# Pipeline-local copies first (as in 05.finetune_dataset/build_11groups.py); ../data/Nadil
# holds a CRLF copy whose different file hash would invalidate existing results.
target_test = first_existing(
    "datasets/finetuning/test.csv",
    "test_dataset_folder/test.csv",
    "datasets/test.csv",
    "../data/Nadil/test.csv",
)

SOURCES = {
    "train": "datasets/finetuning/train.csv",
    "validation": "datasets/finetuning/val.csv",
    "target_test": target_test,
    "mixed_train": "datasets/finetuning/train_mixed_11groups.jsonl",
    "mixed_validation": "datasets/finetuning/val_mixed_11groups.jsonl",
    "dataset_report": "datasets/finetuning/dataset_11groups_report.json",
}
# Benchmarks with the target test set injected (02.preprocess/integrate_test_sets.py)
BENCHMARKS = {
    "commonlid": "datasets/preprocessed/commonlid_integrated.jsonl",
    "flores_plus": "datasets/preprocessed/flores_plus_integrated.jsonl",
    "wili_2018": "datasets/preprocessed/wili-2018_integrated.jsonl",
}

pairs = [(src, c["data"][key]) for key, src in SOURCES.items()]
pairs += [(src, c["data"]["benchmarks"][name]) for name, src in BENCHMARKS.items()]
for src, dst in pairs:
    src_path, dst_path = PIPELINE / src, ROOT / dst
    if not src_path.exists():
        raise FileNotFoundError(f"{src_path} is missing. Run `make preprocess finetune-dataset` first.")
    if dst_path.exists() and digest(dst_path) == digest(src_path):
        status = "unchanged"
    else:
        dst_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(src_path, dst_path)
        status = "copied"
    print(f"{status:>9}  {src} -> new_method/{dst}")

## 2. Build the native fastText continuation binary (NLLB, GlotLID)

In [ ]:
if {"nllb", "glotlid"} & set(models):
    print("Native binary:", build(ROOT))

## 3. Train and evaluate: zero_shot -> target_only, replay

In [ ]:
results = {}
for model in models:
    print(f"===== {model} =====")
    print("Checkpoint:", c["models"][model])
    results[model] = run_experiment(c, model)

## 4. Macro F1 summary (11 groups)

In [ ]:
import json
import pandas as pd

rows = []
for model in models:
    for phase in ("zero_shot", "target_only", "replay"):
        manifest = json.loads((ROOT / c["output_dir"] / model / phase / "complete.json").read_text(encoding="utf8"))
        for benchmark, summary in manifest["benchmark_summaries"].items():
            rows.append({"model": model, "phase": phase, "benchmark": benchmark,
                         "target3": summary["macro_f1_target3"], "replay8": summary["macro_f1_replay8"],
                         "all11": summary["macro_f1_11"]})
summary = pd.DataFrame(rows).pivot_table(index=["model", "benchmark"], columns="phase",
                                         values=["target3", "replay8", "all11"])
display(summary.round(4))